In [9]:
%pip install fsspec huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.6/202.6 kB 4.4 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 20.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 64.8 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 16.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 10.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.4/114.4 kB 21.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 14.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.4/108.4 kB 18.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.7/310.7 kB 43.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.3/87.3 kB 22.8 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.1.2 -> 26.0.1
[notice] To update, run: pip3.11 install --upgrade pip
Note: you may need to restart the kernel to use updated p

In [10]:
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter
from sklearn.model_selection import train_test_split


# Preprocessing
## Import Primary datasets
We will import the main three datasets that we will train out models on.

In [11]:
## Load Mental Health Text Classification Dataset

# path = kagglehub.dataset_download("suchintikasarkar/sentiment-analysis-for-mental-health")
# df_safmh = pd.read_csv(f"{path}/Combined Data.csv", encoding="latin-1", on_bad_lines="skip")

df_mhtc = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS,
                                 "priyangshumukherjee/mental-health-text-classification-dataset",
                                 "mental_health_combined_test.csv")

print(df_mhtc.info())
print(set(df_mhtc['status']))
df_mhtc.head(10)

<class 'pandas.DataFrame'>
RangeIndex: 992 entries, 0 to 991
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    992 non-null    str  
 1   status  992 non-null    str  
dtypes: str(2)
memory usage: 15.6 KB
None
{'Normal', 'Anxiety', 'Suicidal', 'Depression'}


,text,status
0,i don't understand whats wrong with me. i don'...,Anxiety
1,usually when i have anxiety just chatting with...,Anxiety
2,"well, i've had anxiety and panic syndrome for ...",Anxiety
3,"for the most minimal of things, like standing ...",Anxiety
4,i stay away from family and live with my roomm...,Anxiety
5,"i'm very ecstatic, i got a job that literally ...",Anxiety
6,"i've been feeling anxious today, but it's alwa...",Anxiety
7,i quit my job today. it was fucking terrible. ...,Anxiety
8,after months of being stuck in a loop of stayi...,Anxiety
9,i’m at a resort with my friends and their pare...,Anxiety


In [12]:
## Load Sentiment Analysis for Mental Health
df_safmh = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS,
                                "suchintikasarkar/sentiment-analysis-for-mental-health",
                                "Combined Data.csv",
                                pandas_kwargs={"encoding": "latin-1", "on_bad_lines": "skip"})

print(df_safmh.info())
print(set(df_safmh['status']))
df_safmh.head(10)

<class 'pandas.DataFrame'>
RangeIndex: 53043 entries, 0 to 53042
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Unnamed: 0  53043 non-null  int64
 1   statement   52681 non-null  str  
 2   status      53043 non-null  str  
dtypes: int64(1), str(2)
memory usage: 1.2 MB
None
{'Stress', 'Suicidal', 'Normal', 'Anxiety', 'Personality disorder', 'Bipolar', 'Depression'}


,Unnamed: 0,statement,status
0,0,oh my gosh,Anxiety
1,1,"trouble sleeping, confused mind, restless hear...",Anxiety
2,2,"All wrong, back off dear, forward doubt. Stay ...",Anxiety
3,3,I've shifted my focus to something else but I'...,Anxiety
4,4,"I'm restless and restless, it's been a month n...",Anxiety
5,5,"every break, you must be nervous, like somethi...",Anxiety
6,6,"I feel scared, anxious, what can I do? And may...",Anxiety
7,7,Have you ever felt nervous but didn't know why?,Anxiety
8,8,"I haven't slept well for 2 days, it's like I'm...",Anxiety
9,9,"I'm really worried, I want to cry.",Anxiety


In [13]:
## Load Ourafla's Dataset
## Need to combine training and test set before cleaning

df_ourafla_train = pd.read_csv("hf://datasets/ourafla/Mental-Health_Text-Classification_Dataset/mental_heath_unbanlanced.csv")
df_ourafla_test = pd.read_csv("hf://datasets/ourafla/Mental-Health_Text-Classification_Dataset/mental_health_combined_test.csv")

df_ourafla = pd.concat([df_ourafla_train, df_ourafla_test], ignore_index=True)

print(df_ourafla.info())
print(set(df_ourafla['status']))
df_ourafla.head(10)

<class 'pandas.DataFrame'>
RangeIndex: 50604 entries, 0 to 50603
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Unique_ID  40012 non-null  float64
 1   text       50604 non-null  str    
 2   status     50604 non-null  str    
dtypes: float64(1), str(2)
memory usage: 1.2 MB
None
{'Normal', 'Anxiety', 'Suicidal', 'Depression'}


,Unique_ID,text,status
0,0.0,oh my gosh,Anxiety
1,1.0,"trouble sleeping, confused mind, restless hear...",Anxiety
2,2.0,"All wrong, back off dear, forward doubt. Stay ...",Anxiety
3,3.0,I've shifted my focus to something else but I'...,Anxiety
4,4.0,"I'm restless and restless, it's been a month n...",Anxiety
5,5.0,"every break, you must be nervous, like somethi...",Anxiety
6,6.0,"I feel scared, anxious, what can I do? And may...",Anxiety
7,7.0,Have you ever felt nervous but didn't know why?,Anxiety
8,8.0,"I haven't slept well for 2 days, it's like I'm...",Anxiety
9,9.0,"I'm really worried, I want to cry.",Anxiety


## Clean the datasets
We will remove null entries and ensure that we remove entries with statuses other than Anxiety, Suicidal, Normal, or Depression. We need to remove the index from second dataset and rename the 'statement' feature to 'text' to match the other two dataset it will be combined with.

In [14]:
def clean_datasets(data, valid_statuses, index_name=None):
  '''
    Return a cleaned version of the given DataFrame. Remove null entries and standardize status categories.
    Remove index using given index_name

    Params:
      data (DataFrame): given DataFrame to be cleaned
      valid_statuses (List): list of status categories to keep
    Returns:
      data_cleaned (DataFrame): cleaned DataFrame
  '''
  data_cleaned = data.dropna()
  data_cleaned = data_cleaned[data_cleaned['status'].isin(valid_statuses)]
  if index_name is not None:
    data_cleaned = data_cleaned.drop(columns=[index_name])

  return data_cleaned

In [15]:
valid_statuses = ('Suicidal', 'Anxiety', 'Normal', 'Depression')

df_mhtc_cleaned = clean_datasets(df_mhtc, valid_statuses)
df_safmh_cleaned = clean_datasets(df_safmh, valid_statuses, 'Unnamed: 0')
df_ourafla_cleaned = clean_datasets(df_ourafla, valid_statuses, 'Unique_ID')

In [16]:
## Need to rename column for safmh
df_safmh_cleaned = df_safmh_cleaned.rename(columns={'statement': 'text'})
df_safmh_cleaned.info()

<class 'pandas.DataFrame'>
Index: 46240 entries, 0 to 53042
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    46240 non-null  str  
 1   status  46240 non-null  str  
dtypes: str(2)
memory usage: 1.1 MB


## Combine the datasets
We now take the cleaned datasets and combine them because they should have the same statuses and format now. After combining, we make sure to remove duplicates that may appear.

In [17]:
## Check info of main combined dataset
df_main = pd.concat([df_mhtc_cleaned, df_safmh_cleaned, df_ourafla_cleaned], ignore_index=True)
df_main.info()

<class 'pandas.DataFrame'>
RangeIndex: 87244 entries, 0 to 87243
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    87244 non-null  str  
 1   status  87244 non-null  str  
dtypes: str(2)
memory usage: 1.3 MB


In [18]:
## Drop duplicates
df_main.drop_duplicates().info()
df_main = df_main.drop_duplicates()
df_main.head()

## AFTER RUNNING: about 30000 entries dropped.

<class 'pandas.DataFrame'>
Index: 50052 entries, 0 to 87091
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    50052 non-null  str  
 1   status  50052 non-null  str  
dtypes: str(2)
memory usage: 1.1 MB


,text,status
0,i don't understand whats wrong with me. i don'...,Anxiety
1,usually when i have anxiety just chatting with...,Anxiety
2,"well, i've had anxiety and panic syndrome for ...",Anxiety
3,"for the most minimal of things, like standing ...",Anxiety
4,i stay away from family and live with my roomm...,Anxiety


## Split the data
Now that our data has been cleaned, we can split it into a train and test set.
These data sets can be used for all the models that we are training.

In [19]:
df_main_train, df_main_test = train_test_split(df_main, train_size=0.8, test_size=0.2)
df_main_train.info()
df_main_test.info()

<class 'pandas.DataFrame'>
Index: 40041 entries, 3335 to 39623
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    40041 non-null  str  
 1   status  40041 non-null  str  
dtypes: str(2)
memory usage: 938.5 KB
<class 'pandas.DataFrame'>
Index: 10011 entries, 29619 to 6433
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    10011 non-null  str  
 1   status  10011 non-null  str  
dtypes: str(2)
memory usage: 234.6 KB


In [20]:
dir_path = './no_handcrafted/'
df_main_train.to_csv(f'{dir_path}/mental_health_text_train.csv')
df_main_test.to_csv(f'{dir_path}/mental_health_text_test.csv')

# Next Steps
Our data is cleaned, but not yet complete. There is little to be done to improve our BERT model's performance, but we can add handcrafted features to improve the performance of the Logistic Regression models with and without TF-IDF.

These datasets will have the same name, but be labeled with an additional '_features' in the file name.

We may also include an index if we find that it is useful for training and testing.